In [7]:
cd ..

/home/karaman/Documents/bet


In [86]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.churn_label_generator import generate_churn_labels



In [87]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")


In [88]:
df_churn = generate_churn_labels(sessions_silver, config_)
df_churn = (
    df_churn
    .join(
        players_silver.select("player_id"),
        on="player_id",
        how="inner"
    )
)
#df_churn.write.mode("append").parquet("./data/bronze/churn_labels")


In [ ]:
player_snapshot = (players_silver
                   .select('player_id','player_idx','country','age_bucket','device_type','acquisition_channel', 'registration_date', 'risk_segment', 'lifecycle_stage', F.col('updated_balance').alias('current_balance'))
                   .join(df_churn.select('player_id','last_session_date'),
                         on='player_id',
                         how='left')
                         .withColumn('days_since_last_session', 
                                     F.date_diff(F.lit(config_.end_date), F.col('last_session_date')))
                         .drop('player_id', 'reference_date')

)
player_snapshot.show()

+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|player_idx|country|age_bucket|device_type|acquisition_channel|registration_date|risk_segment|lifecycle_stage|   current_balance|last_session_date|days_since_last_session|
+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|      1001|     EN|     18-24|    desktop|          affiliate|       2024-04-01|         low|        engaged|156.82000000000002|       2024-06-30|                      0|
|     10013|     GR|     35-44|    desktop|               paid|       2024-02-26|     unknown|            new|              NULL|             NULL|                   NULL|
|     10014|     EN|     18-24|    desktop|            organic|       2024-03-23|      medium|        engaged|            198.61|       2024